# 实验8：ASC-IR 到 Ascend C 的自动代码生成

## 学习目标

1. 理解 ASC-IR -> Ascend C 的代码生成路径
2. 掌握 Traits / genEmitter / paramTypeLists 的作用
3. 区分自动生成 vs 手写 printOperation
4. 串联 前端 -> IR -> 代码生成 完整链路

## 环境准备

In [ ]:
cd ~/pyasc
!echo "=== 代码生成核心文件 ==="
ls include/ascir/Dialect/Asc/IR/Base.td
ls lib/TableGen/GenOpEmitDefs.cpp
!echo ""
!echo "=== 手写路径示例 ==="
ls lib/Target/AscendC/Basic/ 2>/dev/null || echo "请查看实际目录结构"
!echo ""
!echo "=== 测试文件 ==="
ls test/Target/AscendC/basic/vec_binary.mlir

## 1. 代码生成路径

```
ASC-IR Operation
    -> ascir-translate -mlir-to-ascendc
目标后端（分两路）
    -> 标准 API：自动 GenEmitter（Add/Sub/Mul/Div...）
    -> 不规则 API：手写 printOperation（gather_mask, bilinear_interpolation...）
    ->
Ascend C 源码 (.cpp)
```

> 大多数双目运算走自动路径。

In [ ]:
!ascir-translate --help 2>&1 | grep "mlir-to-ascendc"

## 2. 自动生成开关：genEmitter + Traits

In [ ]:
!grep -n "genEmitter\|AscFunc\|AscMemberFunc\|AscConstructor" include/ascir/Dialect/Asc/IR/Base.td

**自动生成条件：** Operation 带有以下任一 Trait ->
- **AscFunc**：函数形式 -> `AscendC::FuncName(...)`
- **AscMemberFunc**：成员函数 -> `obj.method(...)`
- **AscConstructor**：构造函数形式

双目向量运算带 AscFunc 语义，Add/Sub 走自动路径。

## 3. paramTypeLists 参数映射

In [ ]:
!head -80 lib/TableGen/GenOpEmitDefs.cpp

**三类参数编码：**
- **普通参数** -> 直接输出 `v1, v2, v3`
- **类型推导参数** -> 从 LocalTensor 元素类型推导 `float` -> `<float>`
- **枚举/值参数** -> 进入模板列表 `repeat_times=0` -> `<float, 0>`

这就是 L0/L1 带 `<float, 0>` 而 L2 不带的原因。

## 4. Sub 的代码生成验证

In [ ]:
!grep -A2 "ascendc.sub_l" test/Target/AscendC/basic/vec_binary.mlir

**Sub L0~L3 代码生成对照：**
- L0: `AscendC::Sub<float, 0>(v1, v2, v3, v4, v4, v5);`
- L1: `AscendC::Sub<float, 0>(v1, v2, v3, mask_list, v4, v5);`
- L2: `AscendC::Sub(v1, v2, v3, v4);`
- L3: `v1 = v2.operator-(v3);`

## 5. 手写代码生成路径

手写路径适用于参数结构不规则的 API：

In [ ]:
# 查看手写 API 示例（gather / bilinear_interpolation 等）
ls lib/Target/AscendC/Basic/ 2>/dev/null
!echo ""
!echo "=== 特殊 API 的手写 IR 文件 ==="
ls include/ascir/Dialect/Asc/IR/Basic/OpVecBilinearInterpolation.td
ls include/ascir/Dialect/Asc/IR/Basic/OpVecGather.td
ls include/ascir/Dialect/Asc/IR/Basic/OpVecGatherMask.td

**手写路径适用场景：**
- gather / gather_mask：复杂 mask 构造
- bilinear_interpolation：多参数、水平/垂直双重迭代
- 其他不规则 API

> 工程权衡：少数特殊 API 手写 printOperation 比无限扩展自动模板更务实（YAGNI）。

## 6. 端到端三层对照（串联实验6-7-8）

以 Sub 为例的三级串联：

| 层 | L0 | L2 | L3 |
|----|----|----|-----|
| 前端(实验6) | `create_asc_SubL0Op` | `create_asc_SubL2Op` | (无 L3 Builder) |
| IR(实验7) | `ascendc.sub_l0` | `ascendc.sub_l2` | `ascendc.sub_l3` |
| 目标(实验8) | `Sub<float,0>(...)` | `Sub(v1,v2,v3,v4)` | `operator-(v3)` |

## 总结

1. ASC-IR -> Ascend C 代码生成路径
2. Traits + genEmitter + paramTypeLists 协同
3. 自动生成 vs 手写路径的工程权衡
4. 前端 -> IR -> 代码生成 三级串联 = 完整 API 开发流程